# Day 033 Project Solution — Async Batch Classifier

Concurrent LLM batch processing with `BatchProcessor`, `throttled_gather`, and error envelopes.

In [ ]:
import asyncio
import ollama

async def async_chat(prompt: str, model: str = "llama3.2") -> str:
    client = ollama.AsyncClient()
    response = await client.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"]


import asyncio

async def gather_results(coros: list) -> list:
    return list(await asyncio.gather(*coros))


async def throttled_gather(coros: list, max_concurrent: int) -> list:
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(coro):
        async with sem:
            return await coro
    return list(await asyncio.gather(*[_run(c) for c in coros]))


async def process_batch(
    items: list, async_fn, max_concurrent: int = 3
) -> list[dict]:
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(item):
        async with sem:
            try:
                result = await async_fn(item)
                return {"item": item, "status": "ok",
                        "result": result, "error": None}
            except Exception as e:
                return {"item": item, "status": "error",
                        "result": None, "error": str(e)}
    return list(await asyncio.gather(*[_run(i) for i in items]))


class BatchProcessor:
    def __init__(self, max_concurrent: int = 3, model: str = "llama3.2"):
        self.max_concurrent = max_concurrent
        self.model          = model

    async def process(self, items: list, prompt_fn) -> list[dict]:
        async def _call(item):
            return await async_chat(prompt_fn(item), self.model)
        return await process_batch(items, _call, self.max_concurrent)

    def run(self, items: list, prompt_fn) -> list[dict]:
        return asyncio.run(self.process(items, prompt_fn))

## Action 1 — Verify async_chat Works

In [ ]:
# Single async LLM call (baseline)
reply = await async_chat('Reply with the single word: hello')
print(f'Single call reply: {reply.strip()!r}')
assert isinstance(reply, str) and reply.strip()

## Action 2 — Concurrent Batch with throttled_gather

In [ ]:
import time

PROMPTS = [
    'Name one colour. One word only.',
    'Name one country. One word only.',
    'Name one fruit. One word only.',
    'Name one planet. One word only.',
    'Name one animal. One word only.',
]

start  = time.time()
coros  = [async_chat(p) for p in PROMPTS]
results = await throttled_gather(coros, max_concurrent=3)
elapsed = time.time() - start

print(f'Batch of {len(PROMPTS)} items completed in {elapsed:.2f}s')
for prompt, reply in zip(PROMPTS, results):
    print(f'  {prompt:40} → {reply.strip()}')

assert len(results) == len(PROMPTS)

## Action 3 — BatchProcessor with Error Envelopes

In [ ]:
ITEMS = [
    'The product is amazing!',
    'Terrible, would not recommend.',
    'Pretty decent overall.',
]

bp = BatchProcessor(max_concurrent=3)
batch_results = await bp.process(
    ITEMS,
    lambda x: f"Classify as positive/negative/neutral. One word.\n\n'{x}'",
)

ok_count  = sum(1 for r in batch_results if r['status'] == 'ok')
err_count = sum(1 for r in batch_results if r['status'] == 'error')

print(f'BatchProcessor: {ok_count} ok, {err_count} errors')
for r in batch_results:
    label = r['result'].strip() if r['status'] == 'ok' else f"ERROR: {r['error']}"
    print(f'  {r["item"][:35]:35} → {label}')

assert len(batch_results) == len(ITEMS)
assert ok_count > 0

print('\nBatch complete!')